# 📘 Notebook 14: Decorators
**Expected: ~2 questions out of 65**

---
# 📖 THEORY
---

In [ ]:
# A decorator is a function that takes a function and returns a function
from functools import wraps

def my_decorator(func):
    @wraps(func)  # Preserves function metadata!
    def wrapper(*args, **kwargs):
        print("Before function call")
        result = func(*args, **kwargs)
        print("After function call")
        return result
    return wrapper

@my_decorator  # Same as: greet = my_decorator(greet)
def greet(name):
    '''Greet someone'''
    return f"Hello, {name}!"

print(greet("EPAM"))
print(greet.__name__)  # 'greet' (thanks to @wraps)
print(greet.__doc__)   # 'Greet someone' (thanks to @wraps)

In [ ]:
# Without @wraps, metadata is lost!
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def my_func():
    '''My docstring'''
    pass

print(my_func.__name__)  # 'wrapper' (not 'my_func'!)
print(my_func.__doc__)   # None (lost!)

In [ ]:
# Decorator with arguments (3 levels of nesting!)
from functools import wraps

def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)  # = say_hello = repeat(3)(say_hello)
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("EPAM")  # Prints 3 times

In [ ]:
# Stacked decorators - ORDER MATTERS!
def bold(func):
    @wraps(func)
    def wrapper():
        return f"<b>{func()}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper():
        return f"<i>{func()}</i>"
    return wrapper

@bold
@italic
def greet():
    return "Hello"

print(greet())  # <b><i>Hello</i></b>
# Applied bottom-up: italic first, then bold
# Same as: greet = bold(italic(greet))

In [ ]:
# Class-based decorator
class Timer:
    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        import time
        start = time.time()
        result = self.func(*args, **kwargs)
        end = time.time()
        print(f"{self.func.__name__} took {end-start:.4f}s")
        return result

@Timer
def slow_func():
    import time
    time.sleep(0.1)

slow_func()

In [ ]:
# functools.lru_cache - memoization
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(100))  # Instant! Without cache would be very slow
print(fibonacci.cache_info())  # Shows cache stats

---
# 🔬 DEEP DIVE — Traps & Gotchas
---

In [ ]:
# TRAP 1: Without @wraps metadata is lost
from functools import wraps

def bad_dec(f):
    def wrapper(*a, **k):
        return f(*a, **k)
    return wrapper

def good_dec(f):
    @wraps(f)
    def wrapper(*a, **k):
        return f(*a, **k)
    return wrapper

@bad_dec
def hello1():
    '''Greeting function'''
    pass

@good_dec
def hello2():
    '''Greeting function'''
    pass

print(hello1.__name__)  # 'wrapper' (WRONG!)
print(hello2.__name__)  # 'hello2' (CORRECT!)

In [ ]:
# TRAP 2: Decorator with vs without arguments
# Without args: @decorator -> decorator(func)
# With args: @decorator(x) -> decorator(x)(func)
# DIFFERENT structure!

# Without args (2 levels):
def simple(func):
    def wrapper(): return func()
    return wrapper

# With args (3 levels):
def with_args(arg):
    def decorator(func):
        def wrapper(): return func()
        return wrapper
    return decorator

---
# 🧪 INTERACTIVE EXAMPLES
---

In [ ]:
# What will this print?
def d1(f):
    print("d1 applied")
    def w():
        print("d1 wrapper")
        return f()
    return w

def d2(f):
    print("d2 applied")
    def w():
        print("d2 wrapper")
        return f()
    return w

@d1
@d2
def func():
    return "result"

# What prints during decoration?
# What prints when func() is called?

In [ ]:
# Decoration: "d2 applied" then "d1 applied"
# Because: @d2 applied first (bottom-up), then @d1
print("---")
print(func())
# Call: "d1 wrapper" then "d2 wrapper" then "result"
# Because: d1's wrapper calls d2's wrapper calls original

---
# 💻 EXERCISES
---

In [ ]:
# Exercise 1: Create a timing decorator
import time
from functools import wraps

def timer(func):
    # YOUR CODE HERE: measure execution time
    pass

# Test
@timer
def slow():
    time.sleep(0.1)
    return "done"

# result = slow()  # Should print time taken
# assert result == "done"
# print("\u2705 Exercise 1 passed!")

In [ ]:
# Solution 1
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end-start:.4f}s")
        return result
    return wrapper

@timer
def slow():
    time.sleep(0.1)
    return "done"

result = slow()
assert result == "done"
print("\u2705 Exercise 1 passed!")

In [ ]:
# Exercise 2: Create a call_counter decorator
from functools import wraps

def call_counter(func):
    # YOUR CODE HERE: track how many times func is called
    # func.call_count should be accessible
    pass

# Tests
# @call_counter
# def greet(name):
#     return f"Hi {name}"
#
# greet("A"); greet("B"); greet("C")
# assert greet.call_count == 3
# print("\u2705 Exercise 2 passed!")

In [ ]:
# Solution 2
from functools import wraps

def call_counter(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.call_count += 1
        return func(*args, **kwargs)
    wrapper.call_count = 0
    return wrapper

@call_counter
def greet(name):
    return f"Hi {name}"

greet("A"); greet("B"); greet("C")
assert greet.call_count == 3
print("\u2705 Exercise 2 passed!")

In [ ]:
# Exercise 3: Create a decorator with arguments: @repeat(n)
from functools import wraps

def repeat(n):
    # YOUR CODE HERE: call decorated function n times, return last result
    pass

# Tests
# @repeat(3)
# def say(msg):
#     print(msg, end=" ")
#     return msg
#
# result = say("hi")  # Prints "hi hi hi "
# assert result == "hi"
# print()
# print("\u2705 Exercise 3 passed!")

In [ ]:
# Solution 3
from functools import wraps

def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say(msg):
    print(msg, end=" ")
    return msg

result = say("hi")
assert result == "hi"
print()
print("\u2705 Exercise 3 passed!")

---
# \u26a1 SPEED ROUND
---

In [ ]:
# \u26a1 Q1: @decorator same as?

In [ ]:
print("\u2705 func = decorator(func)")

In [ ]:
# \u26a1 Q2: Stacked @a @b @c order?

In [ ]:
print("\u2705 a(b(c(func)))")

In [ ]:
# \u26a1 Q3: @wraps preserves?

In [ ]:
print("\u2705 __name__, __doc__")

In [ ]:
# \u26a1 Q4: Decorator with args: levels?

In [ ]:
print("\u2705 3 (outer, decorator, wrapper)")

In [ ]:
# \u26a1 Q5: @lru_cache does?

In [ ]:
print("\u2705 Memoization/caching")

In [ ]:
# \u26a1 Q6: Class decorator needs?

In [ ]:
print("\u2705 __init__ + __call__")

In [ ]:
# \u26a1 Q7: @property is decorator?

In [ ]:
print("\u2705 Yes!")

In [ ]:
# \u26a1 Q8: Decorator returns?

In [ ]:
print("\u2705 A function (usually)")

In [ ]:
# \u26a1 Q9: Without @wraps, __name__=?

In [ ]:
print("\u2705 wrapper")

In [ ]:
# \u26a1 Q10: Can decorate class?

In [ ]:
print("\u2705 Yes! (@dataclass)")

---
# 📝 QUIZ (25 Questions)
---

In [ ]:
# Q1 🟡: @a @b @c def f: order?
# A) a(b(c(f)))  B) c(b(a(f)))

In [ ]:
print("✅ A) a(b(c(f)))")
print("📖 Applied bottom-up: c first, then b, then a")

In [ ]:
# Q2 🟡: Without @wraps?
# A) No effect  B) Metadata lost

In [ ]:
print("✅ B) Metadata lost")
print("📖 __name__ and __doc__ become wrapper's")

In [ ]:
# Q3 🟡: Decorator with args has how many layers?
# A) 2  B) 3  C) 1

In [ ]:
print("✅ B) 3")
print("📖 outer(args) -> decorator(func) -> wrapper(*args)")

In [ ]:
# Q4 🟢: @decorator same as?
# A) func=decorator(func)  B) func=decorator()

In [ ]:
print("✅ A) func=decorator(func)")
print("📖 @ is syntactic sugar")

In [ ]:
# Q5 🟡: lru_cache does?
# A) Logging  B) Memoization  C) Timing

In [ ]:
print("✅ B) Memoization")
print("📖 Caches function results")

In [ ]:
# Q6 🟡: Class-based decorator needs?
# A) __init__+__call__  B) __init__ only

In [ ]:
print("✅ A) __init__+__call__")
print("📖 __call__ makes instance callable")

In [ ]:
# Q7 🟢: functools.wraps preserves?
# A) Return value  B) __name__, __doc__

In [ ]:
print("✅ B)")
print("📖 Preserves original function metadata")

In [ ]:
# Q8 🟡: @property is a decorator?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Built-in decorator")

In [ ]:
# Q9 🟡: @staticmethod is a decorator?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Built-in decorator")

In [ ]:
# Q10 🟢: Decorator returns?
# A) None  B) A function  C) A class

In [ ]:
print("✅ B) A function")
print("📖 Returns wrapper function")

In [ ]:
# Q11 🟡: Can stack multiple decorators?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Unlimited stacking")

In [ ]:
# Q12 🟡: @repeat(3) has parens because?
# A) Syntax requirement  B) Calls repeat(3) first

In [ ]:
print("✅ B)")
print("📖 repeat(3) returns the actual decorator")

In [ ]:
# Q13 🟢: Practical use: timing?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Common decorator pattern")

In [ ]:
# Q14 🟡: Practical use: authentication?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Check auth before function runs")

In [ ]:
# Q15 🟡: Decorator can modify return value?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 wrapper can transform result")

In [ ]:
# Q16 🟡: @functools.total_ordering does?
# A) Sorting  B) Auto comparison methods

In [ ]:
print("✅ B)")
print("📖 Only need __eq__ and one comparison")

In [ ]:
# Q17 🟡: @functools.singledispatch does?
# A) Type-based dispatch  B) Caching

In [ ]:
print("✅ A)")
print("📖 Different implementations based on type")

In [ ]:
# Q18 🟢: Decorators are higher-order functions?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Take function as arg, return function")

In [ ]:
# Q19 🟡: Can decorate a class?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Class decorators exist (@dataclass)")

In [ ]:
# Q20 🟡: lru_cache maxsize=None?
# A) No cache  B) Unlimited cache

In [ ]:
print("✅ B) Unlimited cache")
print("📖 Unbounded cache")

In [ ]:
# Q21 🟢: First-class functions needed for decorators?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Functions must be passable as arguments")

In [ ]:
# Q22 🟡: *args, **kwargs in wrapper why?
# A) Speed  B) Accept any arguments

In [ ]:
print("✅ B)")
print("📖 Make wrapper work with any function signature")

In [ ]:
# Q23 🟡: Decorator can add attributes?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Can add call_count etc.")

In [ ]:
# Q24 🟡: @wraps is itself a decorator?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Decorator that preserves metadata")

In [ ]:
# Q25 🔴: Stacked: @bold @italic greet() = ?
# A) bold(italic(greet))  B) italic(bold(greet))

In [ ]:
print("✅ A) bold(italic(greet))")
print("📖 Bottom-up application")

---
# 🔑 KEY TAKEAWAYS
---
1. **Decorator = function that wraps another function** — `@dec` is `func = dec(func)`
2. **Always use `@functools.wraps`** — preserves `__name__`, `__doc__`
3. **Stacked decorators apply bottom-up** — `@a @b def f` = `a(b(f))`
4. **Decorator with arguments** needs 3 nesting levels
5. **`@lru_cache`** for memoization